# Earnalism narration job `narration-7e6b3c2c-c1of2`

**Prepared, not run.** Input zip SHA256 `3f0cefaff6275c5a65267fe825e9009bfee9dcfd050e9074ac71b14346bbb657`. Copy `inputs.zip` to `/content/drive/MyDrive/earnalism-audiovisual/shasti/narration-7e6b3c2c-c1of2/` in Drive, set `RUN = True` in the first cell, then run all. Outputs and `manifest.json` (sha256 of every output) land in the same folder.

**Hard stop:** 3596 s (the reserved forecast) from the clock cell. No item starts after it and the runtime is unassigned at it; a stopped run's manifest says INCOMPLETE and cannot be imported.

**Disconnected?** Run all again: finished items are kept (`progress.jsonl`, checked by sha256) and the run continues where it stopped, within the same reserved time (`sessions.jsonl` counts what earlier runs used).


In [ ]:
RUN = False

In [ ]:
if RUN is not True:
    raise RuntimeError("RUN is False; set RUN = True in the first cell to run this job")
import pathlib
JOB_ID = 'narration-7e6b3c2c-c1of2'
TITLE = 'shasti'
KIND = 'narration'
INPUT_ZIP_SHA256 = '3f0cefaff6275c5a65267fe825e9009bfee9dcfd050e9074ac71b14346bbb657'
MODEL_REPO = 'ai4bharat/indic-parler-tts'
MODEL_REVISION = '7b527af5ee8ed1f9a28d80b19703ed9bb8ba10ca'
STEPS = 1
DRIVE_JOB_DIR = '/content/drive/MyDrive/earnalism-audiovisual/shasti/narration-7e6b3c2c-c1of2'
# Hard wall-clock stop = the forecast seconds reserved on the ledger (R25)
DEADLINE_SECONDS = 3596
MANIFEST_RESERVE_S = 60
PACKAGES = ('parler_tts', 'transformers', 'torch', 'soundfile')


In [ ]:
if RUN is not True:
    raise RuntimeError("RUN is False; set RUN = True in the first cell to run this job")
import hashlib, json, os, threading, time
BOOKKEEPING = ("progress.jsonl", "sessions.jsonl")  # resume records: never outputs


def _rows(name):
    rows = []
    path = OUT_DIR / name
    if path.is_file():
        for line in path.read_text().splitlines():
            try:
                row = json.loads(line)
            except ValueError:
                continue  # a line torn by a disconnect
            if isinstance(row, dict):
                rows.append(row)
    return rows


_APPEND_LOCK = globals().get("_APPEND_LOCK") or threading.Lock()


def _append(name, row):
    # the heartbeat thread and the main thread both append: one writer at a time
    with _APPEND_LOCK:
        path = OUT_DIR / name
        torn = path.is_file() and path.stat().st_size > 0 and path.read_bytes()[-1:] != b"\n"
        with open(path, "a") as f:
            f.write(("\n" if torn else "") + json.dumps(row, sort_keys=True) + "\n")
            f.flush()
            os.fsync(f.fileno())


def session_seconds():
    # started -> seconds, the latest line per session (this one once it has started)
    latest = {}
    for row in _rows("sessions.jsonl"):
        if "started" in row and isinstance(row.get("seconds"), (int, float)):
            latest[row["started"]] = row["seconds"]
    return latest


def note_session():
    _append("sessions.jsonl", {"started": JOB_STARTED, "seconds": round(time.time() - JOB_STARTED, 1)})


def resume_done(names):
    # Finished = present AND matching its progress.jsonl line written for THIS inputs.zip (a line
    # of other inputs proves nothing); a file without one is deleted (redone).
    logged = {}
    for row in _rows("progress.jsonl"):
        if isinstance(row.get("file"), str) and isinstance(row.get("sha256"), str) \
                and row.get("zip") == INPUT_ZIP_SHA256:
            logged[row["file"]] = row["sha256"]
    done = set()
    for name in names:
        path = OUT_DIR / name
        if not path.is_file():
            continue
        if logged.get(name) == hashlib.sha256(path.read_bytes()).hexdigest():
            done.add(name)
        else:
            path.unlink()
            print("redo", name, "(no matching progress line)")
    return done


def record_item(key, name, item_s=None, segments_s=None):
    # item_s: the seconds this item (or its batch) took; a resumed session starts from it.
    # segments_s: a joined multi-voice unit's seconds per segment (the manifest reports them)
    digest = hashlib.sha256((OUT_DIR / name).read_bytes()).hexdigest()
    row = {"key": key, "file": name, "sha256": digest, "zip": INPUT_ZIP_SHA256}
    if isinstance(item_s, (int, float)) and not isinstance(item_s, bool) and item_s >= 0:
        row["item_s"] = round(item_s, 1)
    if segments_s is not None:
        row["segments_s"] = [round(float(x), 3) for x in segments_s]
    _append("progress.jsonl", row)
    note_session()


def last_item_seconds():
    # The latest recorded item (or batch) time, so a resumed session's first item starts only if
    # it can finish before the stop; 0.0 before any item of this job finished.
    times = [r["item_s"] for r in _rows("progress.jsonl") if r.get("zip") == INPUT_ZIP_SHA256
             and isinstance(r.get("item_s"), (int, float))]
    return float(times[-1]) if times else 0.0


def _hard_stop():
    try:
        note_session()
    except Exception as e:  # Drive may have dropped; the runtime must still be released
        print("could not record the session seconds:", e)
    finally:
        runtime.unassign()


def _heartbeat():
    while not HEARTBEAT_STOP.wait(HEARTBEAT_S):
        try:
            note_session()  # a session killed mid-batch (OOM, disconnect) is still charged
        except Exception:
            pass


def attach_session():
    # Run once OUT_DIR is known (after the Drive mount): shorten the stop by what earlier sessions
    # used, charge this session from JOB_STARTED, and start the heartbeat.
    global PRIOR_SECONDS, EFFECTIVE_DEADLINE, STOP_GUARD, SESSION_ATTACHED
    if SESSION_ATTACHED:
        return
    PRIOR_SECONDS = sum(v for k, v in session_seconds().items() if k != JOB_STARTED)
    EFFECTIVE_DEADLINE = DEADLINE_SECONDS - PRIOR_SECONDS
    remaining = EFFECTIVE_DEADLINE - (time.time() - JOB_STARTED)
    if remaining <= 0:
        note_session()
        raise RuntimeError("reserved time used up: close the job with the CU Colab shows")
    STOP_GUARD.cancel()
    STOP_GUARD = threading.Timer(remaining, _hard_stop)
    STOP_GUARD.daemon = True
    STOP_GUARD.start()
    note_session()
    SESSION_ATTACHED = True
    threading.Thread(target=_heartbeat, daemon=True).start()
    print("reserved %d s, %d s used by earlier sessions" % (DEADLINE_SECONDS, PRIOR_SECONDS))


# R25: the hard stop is armed here, right after the config, for the full reserved seconds; the
# verify cell then calls attach_session() to take off what earlier sessions used.
HEARTBEAT_S = globals().get("HEARTBEAT_S", 20)
HEARTBEAT_STOP = globals().get("HEARTBEAT_STOP") or threading.Event()
if "JOB_STARTED" not in globals():
    JOB_STARTED = time.time()
    PRIOR_SECONDS = 0
    EFFECTIVE_DEADLINE = DEADLINE_SECONDS
    SESSION_ATTACHED = False
    from google.colab import runtime
    STOP_GUARD = threading.Timer(DEADLINE_SECONDS, _hard_stop)
    STOP_GUARD.daemon = True
    STOP_GUARD.start()
INCOMPLETE = False


def may_start(last_item_s):
    # Between items: start the next one only if it (and the manifest) fits before the hard stop.
    global INCOMPLETE
    left = EFFECTIVE_DEADLINE - (time.time() - JOB_STARTED)
    if left < last_item_s + MANIFEST_RESERVE_S:
        INCOMPLETE = True
        print("HARD STOP: %.0f s left of the %d s still reserved; no further item starts" % (left, EFFECTIVE_DEADLINE))
        return False
    return True


print("hard stop armed:", DEADLINE_SECONDS, "s from now")


In [ ]:
if RUN is not True:
    raise RuntimeError("RUN is False; set RUN = True in the first cell to run this job")
import hashlib, json, pathlib, zipfile
from google.colab import drive
drive.mount("/content/drive")
OUT_DIR = pathlib.Path(DRIVE_JOB_DIR)
INPUT_ZIP = OUT_DIR / "inputs.zip"
if not INPUT_ZIP.is_file():
    raise RuntimeError("Copy inputs.zip into " + str(OUT_DIR) + " first")
actual = hashlib.sha256(INPUT_ZIP.read_bytes()).hexdigest()
if actual != INPUT_ZIP_SHA256:
    raise RuntimeError("inputs.zip sha256 mismatch: " + actual)
with zipfile.ZipFile(INPUT_ZIP) as z:
    JOB = json.loads(z.read("job.json"))
if JOB["job_id"] != JOB_ID:
    raise RuntimeError("job.json belongs to a different job")
print("inputs verified for", JOB_ID)
attach_session()


In [ ]:
# Narration: one DRY WAV per voice unit with Indic Parler-TTS (revision pinned at build time).
# No pause between units is written here: eav.film.narration_track inserts every pause exactly
# once (R8). A multi-voice unit (cast.json) has segments: each is generated on its own with its own
# voice description and they are joined into the one <unit_key>.wav (join_segments, below); an
# expressive title's unit row also carries its segment gaps and room-tone settings.
if RUN is not True:
    raise RuntimeError("RUN is False; set RUN = True in the first cell to run this job")
import subprocess, sys
subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                "git+https://github.com/huggingface/parler-tts.git", "soundfile"], check=True)
import soundfile as sf, torch
from parler_tts import ParlerTTSForConditionalGeneration
from transformers import AutoTokenizer

device = "cuda" if torch.cuda.is_available() else "cpu"
if device == "cuda":
    dtype = torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16
else:
    dtype = torch.float32
# SDPA attention is faster, but some sub-models (the T5 text encoder) refuse it in some
# transformers versions: fall back to the library default instead of failing the run.
try:
    model = ParlerTTSForConditionalGeneration.from_pretrained(
        MODEL_REPO, revision=MODEL_REVISION, torch_dtype=dtype,
        attn_implementation="sdpa").to(device)
    ATTENTION = "sdpa"
except ValueError as e:
    if "attention implementation" not in str(e) and "attn_implementation" not in str(e):
        raise
    model = ParlerTTSForConditionalGeneration.from_pretrained(
        MODEL_REPO, revision=MODEL_REVISION, torch_dtype=dtype).to(device)
    ATTENTION = "default"
tokenizer = AutoTokenizer.from_pretrained(MODEL_REPO, revision=MODEL_REVISION)
# Parler conditions on the prompt prepended to the audio: a batch must pad prompts on the LEFT
# (the pinned revision's config says so; this holds even if a later revision drops the setting).
tokenizer.padding_side = "left"
if tokenizer.padding_side != "left":
    raise RuntimeError("the prompt tokenizer does not pad on the left")
desc_tokenizer = AutoTokenizer.from_pretrained(
    model.config.text_encoder._name_or_path)
GENERATION = dict(JOB.get("generation", {}))  # pinned roster preset settings (R9)
if GENERATION and "do_sample" not in GENERATION:
    GENERATION["do_sample"] = True  # a temperature only applies when sampling
BATCH = JOB["batch"]  # pinned units per generate() call (part of the job identity)
MODEL_INFO = {"repo": MODEL_REPO, "revision": MODEL_REVISION, "generation": GENERATION,
              "batch": BATCH, "attention": ATTENTION}

import numpy as np
"""Join the separately synthesized segments of one voice unit into the unit's single WAV.

The narration notebook embeds this file's source verbatim (colab_jobs._notebook), so the code the
GPU runs is the code the tests exercise. It needs only numpy. A unit without segments is never
passed through here: its WAV is written exactly as before (dry, nothing added).

Between two segments (a narrator line and a character line, or two speakers) it inserts
SEGMENT_GAP_MS of silence, a short natural breath between voices, and fades the touching edges in
and out over EDGE_FADE_MS so the cut to silence never clicks. The unit's outer start and end are
left untouched, so measured unit ends stay speech ends (film.narration_track adds every pause
between units, once).

Expressive titles (eav.expressive, a cast.json with `tone`) pass each gap's length (gaps_ms, from
the unit's job row) and room-tone settings (room): the gap is then a soft, deterministic room-tone
bed (room_tone) instead of digital silence. film.narration_track uses the same room_tone between
units of such a title.
"""
SEGMENT_GAP_MS = 150
EDGE_FADE_MS = 5


def room_tone(n, rate, dbfs=-62.0, seed=0, fade_ms=10, smooth_ms=0.5):
    """n samples of soft room tone at `dbfs` RMS: splitmix64 noise of the sample index (the same
    on every machine and numpy version), smoothed by a Hann kernel of smooth_ms, with fade_ms
    raised-cosine fades in and out. Deterministic for (n, rate, dbfs, seed, fade_ms, smooth_ms)."""
    import numpy as np
    n = int(n)
    if n <= 0:
        return np.zeros(0, dtype="float32")
    with np.errstate(over="ignore"):
        x = np.arange(n, dtype=np.uint64) * np.uint64(0x9E3779B97F4A7C15) \
            + np.uint64(int(seed) & 0xFFFFFFFFFFFFFFFF)
        x = (x ^ (x >> np.uint64(30))) * np.uint64(0xBF58476D1CE4E5B9)
        x = (x ^ (x >> np.uint64(27))) * np.uint64(0x94D049BB133111EB)
        x = x ^ (x >> np.uint64(31))
    noise = (x >> np.uint64(11)).astype(np.float64) / float(1 << 53) * 2.0 - 1.0
    k = max(1, int(round(rate * smooth_ms / 1000)))
    if k > 1:
        kernel = np.hanning(k + 2)[1:-1]
        noise = np.convolve(noise, kernel / kernel.sum(), mode="same")
    rms = float(np.sqrt(np.mean(noise ** 2)))
    if rms > 0:
        noise = noise * (10 ** (dbfs / 20.0) / rms)
    f = min(int(round(rate * fade_ms / 1000)), n // 2)
    if f:
        ramp = 0.5 - 0.5 * np.cos(np.pi * (np.arange(f) + 0.5) / f)
        noise[:f] *= ramp
        noise[-f:] *= ramp[::-1]
    return noise.astype("float32")


def join_segments(wavs, rate, gap_ms=SEGMENT_GAP_MS, fade_ms=EDGE_FADE_MS, gaps_ms=None, room=None):
    """wavs: the segments' mono samples in order (each already trimmed to its generated length).
    gaps_ms: one gap per pair of segments (else gap_ms for every gap); room: room-tone settings
    {"dbfs", "seed", "fade_ms", "smooth_ms"} (gap k seeded seed + k), else the gaps are silence.
    Returns (joined float32 samples, [seconds of each segment as generated])."""
    import numpy as np
    parts = [np.asarray(w, dtype="float32").reshape(-1).copy() for w in wavs]
    if not parts:
        raise ValueError("a joined unit needs at least one segment")
    seconds = [round(len(p) / float(rate), 3) for p in parts]
    if len(parts) == 1:
        return parts[0], seconds
    gaps = list(gaps_ms) if gaps_ms is not None else [gap_ms] * (len(parts) - 1)
    if len(gaps) != len(parts) - 1:
        raise ValueError(f"{len(parts)} segments need {len(parts) - 1} gaps, not {len(gaps)}")
    fade = int(rate * fade_ms / 1000)
    out = []
    for i, p in enumerate(parts):
        k = min(fade, len(p) // 2)
        if k:
            ramp = (np.arange(k, dtype="float32") + 0.5) / k  # 0 -> 1, never exactly 1 at the edge
            if i > 0:
                p[:k] *= ramp
            if i < len(parts) - 1:
                p[-k:] *= ramp[::-1]
        out.append(p)
        if i < len(parts) - 1:
            n = int(round(rate * gaps[i] / 1000))
            if room:
                out.append(room_tone(n, rate, room.get("dbfs", -62.0), int(room.get("seed", 0)) + i,
                                     room.get("fade_ms", 10), room.get("smooth_ms", 0.5)))
            else:
                out.append(np.zeros(n, dtype="float32"))
    return np.concatenate(out).astype("float32"), seconds


def joined_seconds(segment_seconds, gap_ms=SEGMENT_GAP_MS, gaps_ms=None):
    """The length a joined unit has: its segments plus one gap between each pair (gaps_ms: the
    unit's own gaps, as its job row carries them)."""
    if gaps_ms is not None:
        return sum(segment_seconds) + sum(gaps_ms[:max(0, len(segment_seconds) - 1)]) / 1000
    return sum(segment_seconds) + max(0, len(segment_seconds) - 1) * gap_ms / 1000

rate = model.config.sampling_rate
EXPECTED = [unit["unit_key"] + ".wav" for unit in JOB["units"]]
DONE = resume_done(EXPECTED)  # finished in an earlier session: file present and sha256 logged
todo = [u for u in JOB["units"] if u["unit_key"] + ".wav" not in DONE]
# One generation per piece: a single-voice unit is one piece (as it always was); a multi-voice
# unit is one piece per segment, each with that segment's voice description.
PIECES = []
for u in todo:
    if u.get("segments"):
        PIECES += [(u, n, s["text"], s["description"]) for n, s in enumerate(u["segments"])]
    else:
        PIECES.append((u, None, u["text"], u["description"]))
print(len(DONE), "units already done;", len(todo), "to generate (", len(PIECES),
      "generations) in batches of", BATCH)
pending = {}  # unit_key -> {segment index: samples}, until every segment of the unit is generated
# seconds the previous batch took (from progress.jsonl in a resumed session); the next starts
# only if it fits (R25)
last = last_item_seconds()
for i in range(0, len(PIECES), BATCH):
    if not may_start(last):
        break  # a unit with segments still missing is not written; a resumed session redoes it
    chunk = PIECES[i:i + BATCH]
    started = time.time()
    d = desc_tokenizer([c[3] for c in chunk], return_tensors="pt",
                       padding=True).to(device)
    p = tokenizer([c[2] for c in chunk], return_tensors="pt", padding=True).to(device)
    out = model.generate(input_ids=d.input_ids, attention_mask=d.attention_mask,
                         prompt_input_ids=p.input_ids,
                         prompt_attention_mask=p.attention_mask,
                         return_dict_in_generate=True, **GENERATION)
    lengths = getattr(out, "audios_length", None)
    batch_s = time.time() - started
    wrote = []
    for j, (unit, n, _, _) in enumerate(chunk):
        wav = out.sequences[j].cpu().float().numpy().reshape(-1)
        if lengths is not None:
            wav = wav[:int(lengths[j])]
        else:  # no lengths reported: drop only the padding zeros at the end
            wav = np.trim_zeros(wav, "b")
        key = unit["unit_key"]
        if n is None:
            sf.write(str(OUT_DIR / (key + ".wav")), wav.astype("float32"), rate)
            record_item(key, key + ".wav", item_s=batch_s)
            wrote.append(key)
            continue
        got = pending.setdefault(key, {})
        got[n] = wav
        if len(got) == len(unit["segments"]):
            # expressive titles: the unit's own gaps and room tone (eav.expressive); else 150 ms silence
            joined, seconds = join_segments([got[k] for k in range(len(got))], rate,
                                            gaps_ms=unit.get("segment_gaps_ms"),
                                            room=unit.get("room_tone"))
            sf.write(str(OUT_DIR / (key + ".wav")), joined, rate)
            record_item(key, key + ".wav", item_s=batch_s, segments_s=seconds)
            del pending[key]
            wrote.append(key)
    last = time.time() - started
    print("wrote", wrote, "in %.1f s" % last)


In [ ]:
if RUN is not True:
    raise RuntimeError("RUN is False; set RUN = True in the first cell to run this job")
import hashlib, json, time
import importlib.metadata as _md
outputs = {}
for p in sorted(OUT_DIR.iterdir()):
    if p.is_file() and p.name not in ("manifest.json", "inputs.zip") + BOOKKEEPING:
        outputs[p.name] = hashlib.sha256(p.read_bytes()).hexdigest()
if not outputs:
    raise RuntimeError("no outputs were produced")
packages = {}  # R26: the resolved versions (and VCS commits) that actually ran
for _name in PACKAGES:
    try:
        _dist = _md.distribution(_name)
        _url = json.loads(_dist.read_text("direct_url.json") or "{}")
        _commit = _url.get("vcs_info", {}).get("commit_id")
        packages[_name] = _dist.version + (" @ " + _commit if _commit else "")
    except _md.PackageNotFoundError:
        packages[_name] = None
note_session()
_sessions = session_seconds()
complete = not INCOMPLETE and all(name in outputs for name in EXPECTED)
manifest = {"schema": "earnalism.audiovisual-outputs.v1", "job_id": JOB_ID, "title": TITLE,
            "kind": KIND, "input_zip_sha256": INPUT_ZIP_SHA256, "model": MODEL_INFO,
            "status": "COMPLETE" if complete else "INCOMPLETE",
            "deadline_seconds": DEADLINE_SECONDS,
            "elapsed_seconds": round(sum(_sessions.values()), 1), "sessions": len(_sessions),
            "packages": packages, "outputs": outputs}
# a joined multi-voice narration unit: its seconds per segment, from the latest progress line of
# each output of these inputs (per-segment truncation QC); absent for single-voice jobs
_segments = {}
for _row in _rows("progress.jsonl"):
    if _row.get("zip") == INPUT_ZIP_SHA256 and _row.get("file") in outputs \
            and isinstance(_row.get("segments_s"), list):
        _segments[_row["key"]] = _row["segments_s"]
if _segments:
    manifest["segment_seconds"] = _segments
(OUT_DIR / "manifest.json").write_text(json.dumps(manifest, indent=2, sort_keys=True) + "\n")
print("manifest.json written:", manifest["status"], len(outputs), "files")
if not complete:
    print("INCOMPLETE: import refuses this folder; close the job with the CU Colab shows")
